In [13]:
# Step 1: Import Required Libraries

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.impute import SimpleImputer

In [14]:
# Step 2: Load Dataset

df = pd.read_csv("boston.csv") 
df.head()

,CRIM,ZN,INDUS,CHAS,NOX,RM,AGE,DIS,RAD,TAX,PTRATIO,B,LSTAT,MEDV
0,0.00632,18.0,2.31,0,0.538,6.575,65.2,4.0900,1,296.0,15.3,396.90,4.98,24.0
1,0.02731,0.0,7.07,0,0.469,6.421,78.9,4.9671,2,242.0,17.8,396.90,9.14,21.6
2,0.02729,0.0,7.07,0,0.469,7.185,61.1,4.9671,2,242.0,17.8,392.83,4.03,34.7
3,0.03237,0.0,2.18,0,0.458,6.998,45.8,6.0622,3,222.0,18.7,394.63,2.94,33.4
4,0.06905,0.0,2.18,0,0.458,7.147,54.2,6.0622,3,222.0,18.7,396.90,5.33,36.2


In [15]:
# Step 3: Remove Duplicates & Separate Target

df = df.drop_duplicates()

X = df.drop("MEDV", axis=1)
y = df["MEDV"]

In [16]:
# Step 4: Train–Test Split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [17]:
# Step 5: Preprocessing (Scaling Required for PCA)

num_cols = X.columns

preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]), num_cols)
    ]
)

In [18]:
# Step 6: Apply PCA (95% Variance)

pca = PCA(n_components=0.95, random_state=42)

In [19]:
# Step 7: PCA + Multiple Linear Regression Model

pca_lr_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("pca", pca),
    ("model", LinearRegression())
])

pca_lr_model.fit(X_train, y_train)

y_pred_pca_lr = pca_lr_model.predict(X_test)

In [20]:
# Step 8: Evaluate PCA + Linear Regression

pca_lr_r2 = r2_score(y_test, y_pred_pca_lr)
pca_lr_rmse = np.sqrt(mean_squared_error(y_test, y_pred_pca_lr))
pca_lr_mae = mean_absolute_error(y_test, y_pred_pca_lr)

pca_lr_r2, pca_lr_rmse, pca_lr_mae

(0.596993135111747, np.float64(5.43636073507238), 3.34634986563769)

In [21]:
# Step 9: Random Forest Regression

rf_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ))
])

rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)

In [22]:
# Step 10: Evaluate Random Forest

rf_r2 = r2_score(y_test, y_pred_rf)
rf_rmse = np.sqrt(mean_squared_error(y_test, y_pred_rf))
rf_mae = mean_absolute_error(y_test, y_pred_rf)

rf_r2, rf_rmse, rf_mae

(0.8838893598167612, np.float64(2.918018593121695), 2.0421666666666645)

In [23]:
# Step 11: Compare PCA + LR vs Random Forest

comparison = pd.DataFrame({
    "Model": ["PCA + Linear Regression", "Random Forest Regression"],
    "R2 Score": [pca_lr_r2, rf_r2],
    "RMSE": [pca_lr_rmse, rf_rmse],
    "MAE": [pca_lr_mae, rf_mae]
})

comparison

,Model,R2 Score,RMSE,MAE
0,PCA + Linear Regression,0.596993,5.436361,3.346350
1,Random Forest Regression,0.883889,2.918019,2.042167


In [24]:
# Step 12: Final Model Selection

if rf_r2 > pca_lr_r2:
    print("Random Forest Regression performs better than PCA + Linear Regression.")
else:
    print("PCA + Linear Regression performs better.")

Random Forest Regression performs better than PCA + Linear Regression.
